# Multi-Class Classification (0–4) — Hyperparameter Tuning
Neural net trained to predict the **original diagnosis severity (0, 1, 2, 3, 4)**.
- Output layer: 5-class Softmax
- Grid search over learning rate, hidden size, dropout, and weight decay

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import product
import matplotlib.pyplot as plt

## Load & Preprocess Data

In [ ]:
import ssl, urllib.request, io

url = "https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv"

# macOS Python 3.10 sometimes has SSL cert issues — use unverified context as fallback
try:
    df = pd.read_csv(url)
except Exception:
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=ctx) as r:
        df = pd.read_csv(io.BytesIO(r.read()))

# Strip leading spaces from column names
df.columns = df.columns.str.strip()

# Drop rows with missing values ('?')
df = df.replace('?', np.nan).dropna()

# Keep diagnosis as 0–4 (cast to int)
df['diagnosis'] = df['diagnosis'].astype(int)

print(df['diagnosis'].value_counts().sort_index())
df.head()

## Train / Validation / Test Split

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(df)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

feature_cols = [c for c in df.columns if c != 'diagnosis']

def to_tensors(split):
    X = torch.tensor(split[feature_cols].values.astype(np.float32))
    y = torch.tensor(split['diagnosis'].values.astype(np.int64))
    return X, y

X_train, y_train = to_tensors(train_df)
X_val,   y_val   = to_tensors(val_df)
X_test,  y_test  = to_tensors(test_df)

# Normalize using training stats
mean = X_train.mean(dim=0)
std  = X_train.std(dim=0) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

## Neural Network Definition

In [ ]:
NUM_CLASSES = 5  # diagnosis values: 0, 1, 2, 3, 4

class MultiClassNet(nn.Module):
    def __init__(self, input_dim, hidden_size, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, NUM_CLASSES),  # 5 outputs
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        return self.net(x)

## Training & Evaluation Helpers

In [ ]:
def train_model(model, X_tr, y_tr, X_v, y_v, lr, weight_decay, epochs=100, batch_size=32):
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(torch.log(out + 1e-8), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_losses.append(epoch_loss / len(X_tr))

        model.eval()
        with torch.no_grad():
            val_out  = model(X_v)
            val_loss = criterion(torch.log(val_out + 1e-8), y_v).item()
        val_losses.append(val_loss)

    return train_losses, val_losses


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(dim=1)
    return (preds == y).float().mean().item()

## Hyperparameter Grid Search

In [ ]:
param_grid = {
    'lr':           [1e-2, 1e-3, 1e-4],
    'hidden_size':  [32, 64, 128],
    'dropout':      [0.0, 0.3],
    'weight_decay': [0.0, 1e-4],
}

input_dim = X_train.shape[1]
EPOCHS    = 150

results = []

combos = list(product(*param_grid.values()))
print(f'Testing {len(combos)} hyperparameter combinations...')

for i, (lr, hidden_size, dropout, wd) in enumerate(combos):
    torch.manual_seed(0)
    model = MultiClassNet(input_dim, hidden_size, dropout)
    train_losses, val_losses = train_model(
        model, X_train, y_train, X_val, y_val,
        lr=lr, weight_decay=wd, epochs=EPOCHS
    )
    val_acc = accuracy(model, X_val, y_val)
    results.append({
        'lr': lr, 'hidden_size': hidden_size, 'dropout': dropout,
        'weight_decay': wd, 'val_acc': val_acc,
        'final_val_loss': val_losses[-1],
        'model': model, 'train_losses': train_losses, 'val_losses': val_losses
    })
    if (i + 1) % 6 == 0:
        print(f'  {i+1}/{len(combos)} done')

results.sort(key=lambda r: -r['val_acc'])
print('\nTop 5 configurations (by validation accuracy):')
for r in results[:5]:
    print(f"  lr={r['lr']}, hidden={r['hidden_size']}, dropout={r['dropout']}, "
          f"wd={r['weight_decay']} → val_acc={r['val_acc']:.4f}")

## Best Model — Loss Curves & Test Accuracy

In [ ]:
best = results[0]
print(f"Best config: lr={best['lr']}, hidden={best['hidden_size']}, "
      f"dropout={best['dropout']}, wd={best['weight_decay']}")
print(f"Validation accuracy : {best['val_acc']:.4f}")
print(f"Test accuracy       : {accuracy(best['model'], X_test, y_test):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(best['train_losses'], label='Train Loss')
plt.plot(best['val_losses'],   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.title('Best Model — Training vs Validation Loss (Multi-Class 0–4)')
plt.legend()
plt.tight_layout()
plt.show()

## Per-Class Prediction Distribution (Validation Set)

In [ ]:
best['model'].eval()
with torch.no_grad():
    preds = best['model'](X_val).argmax(dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(y_val.numpy(), bins=5, rwidth=0.8)
axes[0].set_title('True Labels (Val)')
axes[0].set_xlabel('Diagnosis')
axes[1].hist(preds, bins=5, rwidth=0.8, color='orange')
axes[1].set_title('Predicted Labels (Val)')
axes[1].set_xlabel('Diagnosis')
plt.tight_layout()
plt.show()

## Validation Accuracy Across All Configs

In [ ]:
labels = [f"lr={r['lr']}\nh={r['hidden_size']}" for r in results]
accs   = [r['val_acc'] for r in results]

plt.figure(figsize=(14, 4))
plt.bar(range(len(accs)), accs, color='steelblue')
plt.xticks(range(len(accs)), labels, fontsize=6, rotation=45, ha='right')
plt.ylabel('Validation Accuracy')
plt.title('All Hyperparameter Configs — Multi-Class Classifier (0–4)')
plt.tight_layout()
plt.show()